# Analyse du Data Drift — Modèle de scoring crédit

Ce notebook compare deux fenêtres temporelles de données de production :
- **Référence** : trafic simulé sans drift (distributions proches de l'entraînement)
- **Courant** : trafic simulé avec drift (distributions décalées pour simuler une dérive)

Les données sont chargées depuis la base PostgreSQL loggée par l'API.

## 1. Imports et configuration

In [1]:
import warnings
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import HTML, display
from evidently import Dataset, Report
from evidently.presets import DataDriftPreset

import sys
sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from app.database import get_engine

warnings.filterwarnings("ignore")

# Coupure temporelle identifiée par analyse des gaps entre batches
CUTOFF = "2026-04-22 12:01:40+00:00"

# Features suivies (celles décalées par le script de simulation avec --drift)
FEATURES = [
    "DAYS_BIRTH",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "score",
]

REPORT_PATH = Path("reports/drift_report.html")
REPORT_PATH.parent.mkdir(exist_ok=True)

print("Configuration OK")

Configuration OK


## 2. Chargement des données depuis PostgreSQL

In [2]:
engine = get_engine()

cols = ", ".join(f'"{ f}"' for f in FEATURES)

reference = pd.read_sql(
    f"SELECT {cols} FROM predictions WHERE timestamp < %(cutoff)s AND timestamp > '2026-04-22 11:30:00+00:00'",
    engine,
    params={"cutoff": CUTOFF},
)

current = pd.read_sql(
    f"SELECT {cols} FROM predictions WHERE timestamp >= %(cutoff)s",
    engine,
    params={"cutoff": CUTOFF},
)

print(f"Référence : {len(reference)} lignes")
print(f"Courant   : {len(current)} lignes")
display(reference.describe().round(2))

Référence : 200 lignes
Courant   : 100 lignes


,DAYS_BIRTH,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,score
count,200.00,200.00,200.00,200.00,149.00,200.00,156.00,200.00
mean,-15735.50,191021.03,691174.36,28132.92,0.50,0.54,0.54,0.32
std,3292.98,92295.40,465872.70,20818.57,0.18,0.19,0.18,0.15
min,-21885.30,54628.21,67904.03,2666.56,0.07,0.02,0.10,0.06
25%,-18162.15,125083.47,338882.91,12075.58,0.38,0.42,0.42,0.19
50%,-15332.05,172643.54,587721.54,23256.12,0.51,0.54,0.52,0.29
75%,-13392.52,238383.16,889373.14,36805.44,0.62,0.67,0.65,0.41
max,-10276.70,497592.45,2915213.68,132176.80,0.99,0.99,0.99,0.75


## 3. Rapport Evidently — Détection de drift

In [3]:
report = Report([DataDriftPreset()])
result = report.run(
    Dataset.from_pandas(reference),
    Dataset.from_pandas(current),
)

result.save_html(str(REPORT_PATH))
print(f"Rapport exporté : {REPORT_PATH.resolve()}")

Rapport exporté : /home/rapha/ai-engineer/credit-scoring-mlops-step-2/monitoring/reports/drift_report.html


In [4]:
summary = pd.DataFrame({
    "mean_ref":    reference.mean(numeric_only=True).round(3),
    "mean_cur":    current.mean(numeric_only=True).round(3),
    "std_ref":     reference.std(numeric_only=True).round(3),
    "std_cur":     current.std(numeric_only=True).round(3),
})
summary["delta_mean_%"] = ((summary["mean_cur"] - summary["mean_ref"]) / summary["mean_ref"].abs() * 100).round(1)

display(summary)
print(f"\nRapport interactif complet : ouvrir dans un navigateur")
print(REPORT_PATH.resolve())

,mean_ref,mean_cur,std_ref,std_cur,delta_mean_%
DAYS_BIRTH,-15735.501,-20675.372,3292.976,3203.998,-31.4
AMT_INCOME_TOTAL,191021.034,133821.141,92295.403,70670.899,-29.9
AMT_CREDIT,691174.362,746684.008,465872.701,488746.311,8.0
AMT_ANNUITY,28132.920,51206.383,20818.566,35574.608,82.0
EXT_SOURCE_1,0.501,0.319,0.182,0.195,-36.3
EXT_SOURCE_2,0.545,0.382,0.188,0.185,-29.9
EXT_SOURCE_3,0.539,0.352,0.184,0.189,-34.7
score,0.317,0.503,0.149,0.171,58.7



Rapport interactif complet : ouvrir dans un navigateur
/home/rapha/ai-engineer/credit-scoring-mlops-step-2/monitoring/reports/drift_report.html


## 4. Interprétation

### Features en drift

Le script `simulate_production.py --drift` applique les décalages suivants :

| Feature | Décalage simulé | Impact attendu |
|---|---|---|
| `DAYS_BIRTH` | Clients plus âgés (+12 ans en moyenne) | Changement de profil démographique |
| `AMT_INCOME_TOTAL` | Revenus réduits de 20 à 40% | Profils financièrement plus fragiles |
| `AMT_CREDIT` | Durée de crédit allongée (36–96 mois vs 12–72) | Montants empruntés plus élevés |
| `AMT_ANNUITY` | Taux d'annuité plus élevé (5–8.5% vs 2.5–5.5%) | Charge de remboursement plus lourde |
| `EXT_SOURCE_1/2/3` | Scores externes décalés de -0.18 | Historique crédit plus risqué |
| `score` | Score de défaut plus élevé (attendu) | Conséquence directe du drift des features |

### Ce que ça signifie en production

Si ce drift se produisait réellement, l'API recevrait des profils **systématiquement plus risqués** que ceux sur lesquels le modèle a été entraîné. Deux risques :

1. **Sous-estimation du risque** : le modèle a été calibré sur des distributions différentes, ses probabilités de défaut pourraient être sous-estimées pour ces nouveaux profils.
2. **Biais de sélection** : le taux de rejet augmente mécaniquement, ce qui peut fausser les métriques métier.

### Actions recommandées

- **Court terme** : surveiller le taux de rejet et la distribution des scores hebdomadairement.
- **Moyen terme** : si le drift est confirmé sur données réelles, déclencher un ré-entraînement du modèle sur un jeu de données incluant les nouveaux profils.
- **Seuil d'alerte** : un drift détecté sur plus de 3 features clés (`EXT_SOURCE`, `AMT_INCOME_TOTAL`, `DAYS_BIRTH`) doit déclencher une revue manuelle.